In [ ]:
Encoder-only → “understands” text.
Decoder-only → “writes / generates” text.
Encoder-decoder → “reads and writes” text.

## Encoder-Only Pretraining

Architecture: Uses only the encoder stack of a Transformer.

Objective: Masked Language Modeling (MLM)

Randomly mask some tokens in the input.

Train the model to predict the masked tokens using context from both sides.

Use Case:

Text understanding tasks: classification, embeddings, question answering.
Example Models: BERT, RoBERTa, DistilBERT.

In [1]:
# Preprocessing

import re
import html
import string

# --- Cleaning functions ---
def clean_text(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r"<[^>]+>", "", text)       # remove HTML tags
    text = html.unescape(text)                 # decode HTML entities
    text = re.sub(r"\\+", " ", text)          # remove literal backslashes
    text = re.sub(r"[\n\r\t]+", " ", text)    # replace escaped characters with space
    text = re.sub(r"\s+", " ", text)          # collapse multiple spaces
    return text.strip()

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.strip()

def filter_gibberish(text, min_words=5):
    words = text.split()
    return len(words) > min_words

def is_non_toxic(text):
    # Placeholder: keep all for now
    return True

# --- Full preprocessing pipeline ---
def preprocess_texts(texts):
    cleaned_texts = []
    for text in texts:
        text = clean_text(text)
        text = normalize_text(text)
        if filter_gibberish(text) and is_non_toxic(text):
            cleaned_texts.append(text)
    # Deduplicate
    unique_texts = list(dict.fromkeys(cleaned_texts))
    return unique_texts


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

raw_corpus = [
    "Artificial intelligence is the future of technology.",
    "Machine learning helps computers learn from data.",
    "Transformers are state of the art in NLP.",
    "Neural networks can model complex patterns.",
    "Deep learning is a subset of machine learning."
]

# --- Preprocess the text ---
corpus = preprocess_texts(raw_corpus)
print("Preprocessed corpus:", corpus)

# Flatten corpus into a single string (character-level)
flat_corpus = " ".join(corpus)


# Vocabulary
vocab = list(dict.fromkeys(flat_corpus))
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
vocab_size = len(vocab)

# Dataset
class WordDataset(Dataset):
    def __init__(self, words, block_size=5):
        self.block_size = block_size
        self.data = [stoi[w] for w in words]

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]
        return torch.tensor(x), torch.tensor(y)

# dataset = WordDataset(flat_corpus)
# dataloader = DataLoader(dataset, batch_size=4, shuffle=True)


Preprocessed corpus: ['artificial intelligence is the future of technology', 'machine learning helps computers learn from data', 'transformers are state of the art in nlp', 'neural networks can model complex patterns', 'deep learning is a subset of machine learning']


In [32]:
import torch
import torch.nn as nn
import random
from torch.utils.data import Dataset, DataLoader

# Suppose flat_corpus is your cleaned list of words
vocab = list(dict.fromkeys(flat_corpus))
stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for i,w in stoi.items()}
vocab_size = len(vocab)

# ---------------------------
# Dataset with random masking
# ---------------------------
class MaskedDataset(Dataset):
    def __init__(self, words, block_size=5, mask_prob=0.15):
        self.words = words
        self.block_size = block_size
        self.mask_prob = mask_prob
        self.data = [stoi[w] for w in words]

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = x.copy()
        # Randomly mask tokens
        for i in range(len(x)):
            if random.random() < self.mask_prob:
                x[i] = stoi.get("[MASK]", vocab_size)  # add special [MASK] token
        return torch.tensor(x), torch.tensor(y)

# Add [MASK] token
vocab.append("[MASK]")
stoi["[MASK]"] = vocab_size
itos[vocab_size] = "[MASK]"
vocab_size += 1

dataset = MaskedDataset(flat_corpus)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# ---------------------------
# Mini Encoder Model
# ---------------------------
class MiniBERT(nn.Module):
    def __init__(self, vocab_size, n_embd=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, n_embd)
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=n_embd, nhead=2)
        self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=1)
        self.fc = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        x = self.embed(x).permute(1,0,2)  # (seq_len, batch, emb)
        x = self.encoder(x)
        x = x.permute(1,0,2)  # back to (batch, seq_len, emb)
        return self.fc(x)

model = MiniBERT(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# ---------------------------
# Training loop
# ---------------------------
for epoch in range(10):
    total_loss = 0
    for x, y in dataloader:
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 5, Loss: 36.1514
Epoch 10, Loss: 24.4606


In [35]:
# using the learned embeddings

tokens = ["artificial", "intelligence", "is", "the", "stock", "market", "today"]
# add special tokens
tokens = ["[PAD]", "[UNK]"] + tokens  

stoi = {tok: i for i, tok in enumerate(tokens)}
itos = {i: tok for tok, i in stoi.items()}
vocab_size = len(stoi)

print("Vocab size:", vocab_size)

def get_sentence_embedding(model, sentence):
    model.eval()
    words = sentence.lower().translate(str.maketrans("", "", string.punctuation)).split()
    idxs = [stoi.get(w, stoi["[UNK]"]) for w in words]  # use UNK if not found
    x = torch.tensor([idxs], dtype=torch.long)

    with torch.no_grad():
        embeddings = model.embed(x)  # (batch, seq_len, emb_dim)
        sentence_embedding = embeddings.mean(dim=1)
    return sentence_embedding

sentence = "artificial intelligence is transforming technology"
embedding = get_sentence_embedding(model, sentence)
print("Sentence embedding shape:", embedding.shape)
    

Vocab size: 9
Sentence embedding shape: torch.Size([1, 64])


In [29]:
# as you would all know that embedding is a vector representation of your sentence. You can use it for downstream tasks like classification, 
# similarity, or clustering.

In [36]:
# Lets have some labels for sentences (e.g., 0 = tech, 1 = sports, 2 = finance).

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Example sentences and labels
sentences = [
    "Artificial intelligence is transforming technology",
    "The football match was exciting",
    "Stock markets are volatile today"
]
labels = [0, 1, 2]

# Generate embeddings for all sentences
X = torch.vstack([get_sentence_embedding(model, s) for s in sentences]).numpy()
y = labels

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Simple classifier on top of embeddings
clf = LogisticRegression()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Test predictions:", y_pred)
# print("Accuracy:", accuracy_score(y_test, y_pred)) # will improve based on more data


Test predictions: [2]
Accuracy: 0.0


In [44]:
#!pip install --upgrade pip setuptools wheel

!pip install --only-binary=:all: pyarrow datasets

  Using cached datasets-4.1.1-py3-none-any.whl.metadata (18 kB)
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached xxhash-3.5.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached multiprocess-0.70.16-py310-none-any.whl.metadata (7.2 kB)
  Using cached huggingface_hub-0.35.0-py3-none-any.whl.metadata (14 kB)
  Using cached aiohttp-3.12.15-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.7 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached frozenlist-1.7.0-cp310-cp310-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
  Using cached multidict-6.6.4-cp310-cp310-manylinux2014_x86_64.manylinux_2_17

# Training on a larger dataset

In [45]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import re, string

# --------------------------
# Data cleaning functions
# --------------------------
def clean_text(text):
    text = re.sub(r"<[^>]+>", "", text)  # remove HTML
    text = text.encode("ascii", "ignore").decode()  # remove special chars
    return text

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)  # normalize spaces
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    return text.strip()

# --------------------------
# Load a small dataset
# --------------------------
dataset = load_dataset("ag_news", split="train[:500]")  # small subset
texts = [normalize_text(clean_text(x["text"])) for x in dataset]

# --------------------------
# Build vocab
# --------------------------
all_words = set(" ".join(texts).split())
special_tokens = ["[PAD]", "[UNK]", "[MASK]"]
vocab = special_tokens + sorted(list(all_words))

stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for tok, i in stoi.items()}
vocab_size = len(stoi)

print("Vocab size:", vocab_size)

# --------------------------
# Dataset class (MLM style)
# --------------------------
class TextDataset(Dataset):
    def __init__(self, texts, stoi, mask_prob=0.15):
        self.texts = texts
        self.stoi = stoi
        self.mask_prob = mask_prob

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        words = self.texts[idx].split()
        ids = [self.stoi.get(w, self.stoi["[UNK]"]) for w in words]

        # Apply random masking for pretraining
        labels = ids.copy()
        for i in range(len(ids)):
            if torch.rand(1).item() < self.mask_prob:
                ids[i] = self.stoi["[MASK]"]
            else:
                labels[i] = -100  # ignore in loss

        return torch.tensor(ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

dataset = TextDataset(texts, stoi)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=lambda batch: (
    nn.utils.rnn.pad_sequence([b[0] for b in batch], batch_first=True, padding_value=stoi["[PAD]"]),
    nn.utils.rnn.pad_sequence([b[1] for b in batch], batch_first=True, padding_value=-100)
))

# --------------------------
# Builiding a MiniBERT (encoder-only)
# --------------------------
class MiniBERT(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden=128, n_layers=2, n_heads=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=n_heads, dim_feedforward=hidden)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.fc = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        emb = self.embed(x)  # (batch, seq, emb_dim)
        emb = emb.transpose(0, 1)  # (seq, batch, emb_dim) for Transformer
        enc = self.encoder(emb)    # (seq, batch, emb_dim)
        enc = enc.transpose(0, 1)  # back to (batch, seq, emb_dim)
        return self.fc(enc)

model = MiniBERT(vocab_size)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --------------------------
# Training loop
# --------------------------
for epoch in range(3): 
    total_loss = 0
    for x, y in dataloader:
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# --------------------------
# Sentence embedding function
# --------------------------
def get_sentence_embedding(model, sentence):
    model.eval()
    words = sentence.lower().translate(str.maketrans("", "", string.punctuation)).split()
    idxs = [stoi.get(w, stoi["[UNK]"]) for w in words]
    x = torch.tensor([idxs], dtype=torch.long)
    with torch.no_grad():
        embeddings = model.embed(x)  # (batch, seq, emb_dim)
        sentence_embedding = embeddings.mean(dim=1)
    return sentence_embedding

# --------------------------
# Example usage
# --------------------------
sentence = "artificial intelligence is transforming technology"
embedding = get_sentence_embedding(model, sentence)
print("Sentence embedding shape:", embedding.shape)


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Vocab size: 5739


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1, Loss: 265.7371
Epoch 2, Loss: 247.9564
Epoch 3, Loss: 245.1147
Sentence embedding shape: torch.Size([1, 64])


In [50]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np


raw_dataset = load_dataset("ag_news", split="train[:200]")  # small subset

X = []
y = []
for example in raw_dataset:
    emb = get_sentence_embedding(model, example["text"])
    if emb is not None:
        X.append(emb.squeeze(0).numpy())
        y.append(example["label"])

X = np.array(X)
y = np.array(y)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a simple classifier
clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.675


# From Hugging Face

In [52]:
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 200.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 151.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers] [transformers]


In [53]:
# Example 1: Masked Language Modeling (MLM) with BERT


from transformers import pipeline

# Load pretrained BERT-base (encoder-only)
unmasker = pipeline("fill-mask", model="bert-base-uncased")

# Example sentences
print(unmasker("Artificial intelligence is [MASK]."))
print(unmasker("The stock market [MASK] today."))


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cpu


[{'score': 0.05532136559486389, 'token': 2825, 'token_str': 'possible', 'sequence': 'artificial intelligence is possible.'}, {'score': 0.0461326502263546, 'token': 2590, 'token_str': 'important', 'sequence': 'artificial intelligence is important.'}, {'score': 0.019756034016609192, 'token': 8050, 'token_str': 'fundamental', 'sequence': 'artificial intelligence is fundamental.'}, {'score': 0.01718037948012352, 'token': 2204, 'token_str': 'good', 'sequence': 'artificial intelligence is good.'}, {'score': 0.015328076668083668, 'token': 2691, 'token_str': 'common', 'sequence': 'artificial intelligence is common.'}]
[{'score': 0.3477132320404053, 'token': 4247, 'token_str': 'continues', 'sequence': 'the stock market continues today.'}, {'score': 0.10261214524507523, 'token': 6526, 'token_str': 'exists', 'sequence': 'the stock market exists today.'}, {'score': 0.053825944662094116, 'token': 5748, 'token_str': 'operates', 'sequence': 'the stock market operates today.'}, {'score': 0.02959034033

In [54]:
# Example 2: Sentence Classification with BERT

from transformers import pipeline

classifier = pipeline("text-classification", model="textattack/bert-base-uncased-ag-news")

text = "Artificial intelligence is transforming technology"
print(classifier(text))

# LABEL_0 → World
# LABEL_1 → Sports
# LABEL_2 → Business
# LABEL_3 → Sci/Tech


config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


[{'label': 'LABEL_0', 'score': 0.512243926525116}]


In [57]:
label_map = {
    "LABEL_0": "World",
    "LABEL_1": "Sports",
    "LABEL_2": "Business",
    "LABEL_3": "Sci/Tech"
}

output = classifier(text)
pred_label = label_map[output[0]['label']]
score = output[0]['score']

print(f"Predicted class: {pred_label}, Score: {score:.2f}")

Predicted class: World, Score: 0.51


In [55]:
# Example 3: Sentence Embeddings (for downstream tasks)

# from sentence_transformers import SentenceTransformer

# # Load a pretrained sentence encoder
# model = SentenceTransformer("all-MiniLM-L6-v2")

# sentences = [
#     "Artificial intelligence is transforming technology",
#     "The stock market fell today",
# ]

# embeddings = model.encode(sentences)
# print("Embedding shape:", embeddings.shape)

# the output would be just an embeddings, which can be used for other purposes


ModuleNotFoundError: No module named 'sentence_transformers'

# Encoder- Decoder 

In [58]:
# BART for text summarization

from transformers import pipeline

# Load pretrained BART model (encoder-decoder)
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

text = """
Artificial intelligence is transforming technology. 
It is being applied in healthcare, finance, autonomous vehicles, and many other domains. 
Companies are investing heavily in AI research to stay competitive.
"""

summary = summarizer(text, max_length=50, min_length=20, do_sample=False)
print(summary)


# Encoder: Reads and encodes the input text into a vector representation.
#Decoder: Generates a new sequence (summary) based on the encoded input.

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu
Your max_length is set to 50, but your input_length is only 44. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=22)


[{'summary_text': 'Artificial intelligence is transforming technology. It is being applied in healthcare, finance, autonomous vehicles, and many other domains.'}]


In [ ]:
# BART for text generation

from transformers import BartTokenizer, BartForConditionalGeneration
import torch

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large')

prompt = "Artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate continuation
outputs = model.generate(**inputs, max_length=30)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
| Model Type          | Example Models | Pretraining Task                                            | Goal                                                        | Use Cases                                       |
| ------------------- | -------------- | ----------------------------------------------------------- | ----------------------------------------------------------- | ----------------------------------------------- |
| **Encoder-Only**    | BERT           | Masked Language Modeling (MLM) ± Next Sentence Prediction   | Learn bidirectional text representations                    | Classification, embeddings, question answering  |
| **Decoder-Only**    | GPT            | Next-Token Prediction (autoregressive LM)                   | Learn to generate text given previous context               | Text generation, completion, conversation, code |
| **Encoder-Decoder** | BART, T5       | Denoising autoencoding (mask/shuffle/corrupt → reconstruct) | Learn both understanding (encoder) and generation (decoder) | Summarization, translation, seq2seq tasks       |
